In [1]:
"""
Task 1 - Data Collection from HackerNews API
Fetches trending stories and saves them as JSON
"""
 
import requests
import json
import time
from datetime import datetime
import os
 
# Categories and their keywords
tech_words = ['AI', 'software', 'tech', 'code', 'computer', 'data', 'cloud', 'API', 'GPU', 'LLM']
news_words = ['war', 'government', 'country', 'president', 'election', 'climate', 'attack', 'global']
sports_words = ['NFL', 'NBA', 'FIFA', 'sport', 'game', 'team', 'player', 'league', 'championship']
science_words = ['research', 'study', 'space', 'physics', 'biology', 'discovery', 'NASA', 'genome']
entertainment_words = ['movie', 'film', 'music', 'Netflix', 'game', 'book', 'show', 'award', 'streaming']
 
# API URLs
base_url = "https://hacker-news.firebaseio.com/v0/"
headers = {"User-Agent": "TrendPulse/1.0"}

def get_category(title):
    # Check what category a story belongs to based on title
    if title is None:
        return None
    
    title = title.lower()
    
    # Check technology keywords
    for word in tech_words:
        if word.lower() in title:
            return 'technology'
    
    # Check worldnews keywords
    for word in news_words:
        if word.lower() in title:
            return 'worldnews'
    
    # Check sports keywords
    for word in sports_words:
        if word.lower() in title:
            return 'sports'
    
    # Check science keywords
    for word in science_words:
        if word.lower() in title:
            return 'science'
    
    # Check entertainment keywords
    for word in entertainment_words:
        if word.lower() in title:
            return 'entertainment'
    
    return None

# Get top story IDs
print("Fetching top stories from HackerNews...")
try:
    response = requests.get(base_url + "topstories.json", headers=headers)
    all_ids = response.json()
    top_500_ids = all_ids[:500]  # Get first 500
    print(f"Got {len(top_500_ids)} story IDs")
except Exception as e:
    print(f"Failed to fetch top stories: {e}")
    exit()

# Store collected stories
stories = []
 
# Keep track of how many stories per category
tech_count = 0
news_count = 0
sports_count = 0
science_count = 0
entertainment_count = 0
 
# Fetch each story
print("\nFetching story details...")
for story_id in top_500_ids:
    # Check if we have enough stories
    if (tech_count >= 25 and news_count >= 25 and sports_count >= 25 
        and science_count >= 25 and entertainment_count >= 25):
        break

    try:
        # Get story details
        story_url = base_url + f"item/{story_id}.json"
        r = requests.get(story_url, headers=headers)
        story_data = r.json()
        
        # Skip if not a story
        if story_data.get('type') != 'story':
            continue
        
        # Get the category
        title = story_data.get('title', '')
        category = get_category(title)
        
        # Skip if no category found
        if category is None:
            continue
        
        # Check if this category is full
        if category == 'technology' and tech_count >= 25:
            continue
        elif category == 'worldnews' and news_count >= 25:
            continue
        elif category == 'sports' and sports_count >= 25:
            continue
        elif category == 'science' and science_count >= 25:
            continue
        elif category == 'entertainment' and entertainment_count >= 25:
            continue
        
        # Extract the fields we need
        story = {
            'post_id': story_data.get('id'),
            'title': title,
            'category': category,
            'score': story_data.get('score', 0),
            'num_comments': story_data.get('descendants', 0),
            'author': story_data.get('by', ''),
            'collected_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        
        stories.append(story)
        
        # Update the count for this category
        if category == 'technology':
            tech_count += 1
        elif category == 'worldnews':
            news_count += 1
        elif category == 'sports':
            sports_count += 1
        elif category == 'science':
            science_count += 1
        elif category == 'entertainment':
            entertainment_count += 1
        
        # Print progress
        if len(stories) % 10 == 0:
            print(f"Collected {len(stories)} stories...")
    
    except Exception as e:
        # If there's an error, print message and skip this story
        print(f"Failed to fetch story {story_id}, skipping...")
        continue
 
# Sleep 2 seconds between categories as required
time.sleep(2)
 
# Create data folder if needed
if not os.path.exists('data'):
    os.makedirs('data')
 
# Save to JSON file
today = datetime.now().strftime('%Y%m%d')
filename = f'data/trends_{today}.json'
 
with open(filename, 'w') as f:
    json.dump(stories, f, indent=2)
 
print(f"\nCollected {len(stories)} stories. Saved to {filename}")
 
# Print breakdown
print(f"\nBreakdown:")
print(f"Technology: {tech_count}")
print(f"World News: {news_count}")
print(f"Sports: {sports_count}")
print(f"Science: {science_count}")
print(f"Entertainment: {entertainment_count}")






Fetching top stories from HackerNews...
Got 500 story IDs

Fetching story details...
Collected 10 stories...
Collected 20 stories...
Collected 30 stories...
Collected 40 stories...
Collected 50 stories...
Collected 60 stories...
Collected 70 stories...

Collected 77 stories. Saved to data/trends_20260405.json

Breakdown:
Technology: 25
World News: 12
Sports: 6
Science: 9
Entertainment: 25
